In [1]:
import pandas as pd

In [23]:
def get_enhancers(file):    
    # Load the data into a DataFrame
    df = pd.read_csv(file, sep=',')
    # Split each pair of peaks into two sets of separate columns
    peaks_split = df['Peak1'].str.split('-', expand=True)
    chr = peaks_split[0].str.split(':', expand=True)[0]
    start_position = peaks_split[0].str.split(':', expand=True)[1]
    end_position = peaks_split[1]
    # peaks_split.columns = ['Peak1_start', 'Peak1_end']

    peaks_split2 = df['Peak2'].str.split('-', expand=True)
    chr2 = peaks_split2[0].str.split(':', expand=True)[0]
    start_position2 = peaks_split2[0].str.split(':', expand=True)[1]
    end_position2 = peaks_split2[1]

    # Vertically concatenate the two sets of columns
    new_df = pd.DataFrame({
        'chr': pd.concat([chr, chr2], ignore_index=True),
        'start_position': pd.concat([start_position, start_position2], ignore_index=True),
        'end_position': pd.concat([end_position, end_position2], ignore_index=True)
    })
    new_df['start_position'] = new_df['start_position'].astype(int)
    new_df['end_position'] = new_df['end_position'].astype(int)
    return new_df

In [24]:
new_df = get_enhancers('data/chr2_coaccess_score_gt0.2.csv')

In [25]:
new_df.start_position

0         10047212
1         10079297
2         10079297
3        101139975
4        101295315
           ...    
37695     98832099
37696     98662054
37697     98832099
37698     98662054
37699     98666048
Name: start_position, Length: 37700, dtype: int32

In [21]:
import random
def get_nearby_enhancers(df,start_position,end_position):
    filtered_index = (df['start_position'] > start_position - 500000) & (df['end_position'] < end_position + 500000)
    enhancer = df[filtered_index]
    # Gnerate random barcode
    def generate_barcode():
        return ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789', k=10))
    # Create a new DataFrame with the required columns
    bed_df = pd.DataFrame({
        'Chromosome': enhancer['chr'],
        'Start': enhancer['start_position'],
        'End': enhancer['end_position'],
        'Barcode': [generate_barcode() for _ in range(len(enhancer))]
    })
    # Reorder columns to match the BED format
    bed_df = bed_df[['Chromosome', 'Start', 'End', 'Barcode']]

    return bed_df

# Example
nearby_enhancers = get_nearby_enhancers(new_df, 140395308, 140395595)
nearby_enhancers

,Chromosome,Start,End,Barcode
2718,chr2,139948460,139949150,GoZ2Qhy2w0
2719,chr2,140113706,140114418,KapjzBBADa
2720,chr2,140113706,140114418,e1OlIQRuiD
2721,chr2,140214370,140216328,hrwcJBje7Y
2722,chr2,140214370,140216328,QthjleSrMw
2723,chr2,140214370,140216328,7FZtY6RXZF
2724,chr2,140214370,140216328,FYUUdcpzCT
2725,chr2,140230150,140230797,AxFA5YIXGz
2726,chr2,140230150,140230797,jfAcUC1jMf
2727,chr2,140253904,140254306,zYhkzO6eVJ


In [ ]:
import pandas as pd
from pybedtools import BedTool

bed_tool = BedTool.from_dataframe(nearby_enhancers)
bed_tool.saveas("output.bed")